In [1]:
import math
import numbers
import os
import sys
from pathlib import Path
from types import SimpleNamespace

import kagglehub
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pv
import scipy.sparse as sp
import torch
from IPython.display import display

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "lightgcl.py").is_file())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from lightgcl import LightGCLTimeSoftmax  # noqa: E402
from utils import normalize_adj, svd_view, to_torch_sparse, topk_metrics  # noqa: E402

DATA_DIR = Path(
    os.environ.get("EXPEDIA_DATA_DIR")
    or kagglehub.competition_download("expedia-hotel-recommendations")
)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def fmt_num(x, digits=4):
    """37670293 -> '37.670.293', 0.12345 -> '0,1235' (at most `digits` decimals)."""
    text = f"{x:,.0f}" if float(x).is_integer() else f"{x:,.{digits}f}".rstrip("0").rstrip(".")
    return text.translate(str.maketrans(",.", ".,"))


def fmt_table(df):
    """Display copy of df with numbers in Vietnamese format."""
    def cell(value):
        if isinstance(value, bool) or not isinstance(value, numbers.Number):
            return value
        return "—" if pd.isna(value) else fmt_num(value)

    out = df.astype(object)
    for col in df.columns:
        out[col] = [cell(value) for value in df[col]]
    return out

In [2]:
cfg = SimpleNamespace(
    item_cols=["hotel_cluster"],
    val_start="2014-09-01",
    test_start="2014-11-01",
    test_end="2015-01-01",
    embed_dim=64,
    n_layers=2,
    temp=0.2,
    svd_q=2,
    svd_exact=True,
    lambda_1=0.2,
    lambda_2=1e-4,
    dropout=0.0,
    topks=(5, 10, 20),
    seed=2026,
    # LightGCL + time attention trained in LightGCL_Click.ipynb, whose model.pt holds one state per variant
    run="lightgcl_click_20261004_115153",
    variant="LightGCL, booking theo số lần + thời gian (GraphPro), λ2 = 1e-4",
    user_ids=[],  # users to check; empty: n_random users drawn among the val users the model knows
    n_random=5,
    min_labels=1,  # random draw only: users who booked at least this many clusters in val
    sample_seed=0,
    show_top=10,  # length of the predicted lists shown
)

In [3]:
COLUMN_TYPES = {"date_time": pa.timestamp("s"), "user_id": pa.int32(), "is_booking": pa.int8(),
                **{c: pa.int32() for c in cfg.item_cols}}
events = pv.read_csv(
    DATA_DIR / "train.csv",
    convert_options=pv.ConvertOptions(include_columns=list(COLUMN_TYPES), column_types=COLUMN_TYPES),
).to_pandas()
val_start, test_start, test_end = (pd.Timestamp(d) for d in (cfg.val_start, cfg.test_start, cfg.test_end))
events = events[events["date_time"] < test_end]
events["item"] = events.groupby(cfg.item_cols, sort=True).ngroup().astype(np.int32)
assert (events["item"] == events["hotel_cluster"]).all()  # so the item index of the model is the hotel_cluster id
n_items = events["item"].nunique()
bookings = events[events["is_booking"] == 1]


def booking_graph(end):
    """Graph of the bookings before `end`, built as in LightGCL_Final.ipynb: train_csr holds the booked pairs, adj_csr
    the same pairs weighted by their number of bookings."""
    rows = bookings[bookings["date_time"] < end]
    user_index = pd.Index(np.sort(rows["user_id"].unique()), name="user_id")
    pair_counts = rows.groupby([user_index.get_indexer(rows["user_id"]), rows["item"]]).size()
    book_counts = sp.csr_matrix(
        (pair_counts.to_numpy(np.float32),
         (pair_counts.index.get_level_values(0), pair_counts.index.get_level_values(1))),
        shape=(len(user_index), n_items),
    )
    return SimpleNamespace(end=end, user_index=user_index, n_users=len(user_index),
                           train_csr=(book_counts > 0).astype(np.float32).tocsr(), adj_csr=book_counts)


def make_split(start, end):
    """Labels of the bookings in [start, end): user ids and one row per user, one column per cluster."""
    pairs = bookings[bookings["date_time"].between(start, end, inclusive="left")][["user_id", "item"]].drop_duplicates()
    users, row = np.unique(pairs["user_id"].to_numpy(), return_inverse=True)
    return SimpleNamespace(
        user_ids=users,
        labels=sp.csr_matrix((np.ones(len(pairs), np.float32), (row, pairs["item"].to_numpy())),
                             shape=(len(users), n_items)),
    )


def val_labels(user_ids):
    """Set of the clusters each user booked in val, the labels the metrics are computed against."""
    rows, labels = np.searchsorted(val_split.user_ids, user_ids), val_split.labels
    return [set(labels.indices[labels.indptr[r]:labels.indptr[r + 1]].tolist()) for r in rows]


graph = booking_graph(val_start)  # the graph the model was trained on: bookings before 09/2014
val_split = make_split(val_start, test_start)
val_users = val_split.user_ids[np.isin(val_split.user_ids, graph.user_index)]  # the val users scored during training
print(f"{fmt_num(len(val_split.user_ids))} user book trong val (09–10/2014), {fmt_num(len(val_users))} user trong số đó "
      f"đã book trước 09/2014 (model biết, được chấm val lúc train)")

226.277 user book trong val (09–10/2014), 148.736 user trong số đó đã book trước 09/2014 (model biết, được chấm val lúc train)


## 1. User và các cluster họ book trong val

In [5]:
if cfg.user_ids:
    users = np.asarray(cfg.user_ids)
    dropped = users[~np.isin(users, val_users)]
    if len(dropped):
        print(f"Bỏ {dropped.tolist()}: không book trong val hoặc chưa book trước 09/2014 (model không chấm)")
    users = users[np.isin(users, val_users)]
else:
    pool = val_users[np.array([len(s) for s in val_labels(val_users)]) >= cfg.min_labels]
    users = np.sort(np.random.default_rng(cfg.sample_seed).choice(pool, cfg.n_random, replace=False))
labels = val_labels(users)


def counted(items):
    """Clusters of a series of bookings, most booked first: '91 (×3), 41'."""
    return ", ".join(f"{c} (×{n})" if n > 1 else str(c) for c, n in items.value_counts().items())


mine = bookings[bookings["user_id"].isin(users)]
past = mine[mine["date_time"] < val_start].groupby("user_id")["item"]
in_val = mine[mine["date_time"].between(val_start, test_start, inclusive="left")].groupby("user_id")["item"]
display(pd.DataFrame({
    "booking trước 09/2014": past.size(),
    "cluster đã book trước 09/2014": past.apply(counted),
    "booking trong val": in_val.size(),
    "cluster book trong val": in_val.apply(counted),
}).loc[users].rename_axis("user_id"))

,booking trước 09/2014,cluster đã book trước 09/2014,booking trong val,cluster book trong val
user_id,,,,
329118,6,"72 (×2), 68, 41, 95, 38",1,38
373861,5,"28 (×2), 47, 97, 67",1,55
618548,5,"98 (×2), 48, 91, 9",2,"49, 42"
775664,4,"86, 75, 59, 64",1,97
1029342,2,"91, 42",2,"94, 42"


## 2. Model dự đoán

In [6]:
def time_attention(graph):
    """Time attention weights of a graph, as in LightGCL_Final.ipynb: the softmax over each user's clusters (time_u,
    users x items) and over each cluster's users (time_i, items x users) of the pair's last booking time, scaled to
    [0, 1] over the graph."""
    rows = bookings[bookings["date_time"] < graph.end]
    last = rows.groupby([graph.user_index.get_indexer(rows["user_id"]), rows["item"].to_numpy()])["date_time"].max()
    scaled = ((last - last.min()) / (last.max() - last.min())).to_numpy(np.float64)
    exp_t = sp.csr_matrix((np.exp(scaled), (last.index.get_level_values(0), last.index.get_level_values(1))),
                          shape=(graph.n_users, n_items))
    time_u = sp.diags(1 / np.asarray(exp_t.sum(1)).ravel()) @ exp_t
    time_i = (exp_t @ sp.diags(1 / np.asarray(exp_t.sum(0)).ravel())).T
    return time_u, time_i


def make_model(graph):
    """LightGCL + time attention on graph, as in LightGCL_Final.ipynb."""
    adj_norm = to_torch_sparse(normalize_adj(graph.adj_csr), DEVICE)
    torch.manual_seed(cfg.seed)
    factors = svd_view(adj_norm, cfg.svd_q, cfg.svd_exact)
    time_u, time_i = (to_torch_sparse(m.tocoo(), DEVICE) for m in time_attention(graph))
    return LightGCLTimeSoftmax(graph.n_users, n_items, cfg.embed_dim, *factors, graph.train_csr, adj_norm, cfg.n_layers,
                               cfg.temp, cfg.lambda_1, cfg.lambda_2, cfg.dropout, time_u=time_u, time_i=time_i).to(DEVICE)


# mmap: only the state of cfg.variant is read from the 9 GB file
saved = torch.load(PROJECT_ROOT / "data" / "runs" / cfg.run / "model.pt", map_location="cpu", mmap=True,
                   weights_only=False)
state, state_users, trained = (saved[key][cfg.variant] for key in ("states", "user_ids", "results"))
del saved
assert np.array_equal(state_users, graph.user_index), "the model was trained on another list of users"
model = make_model(graph)
model.load_state_dict(state)
print(f"Model: {cfg.variant} (data/runs/{cfg.run}), epoch {trained['best_epoch']}")


def predict(user_ids):
    """Scores (users x clusters) as in the val evaluation during training: model.predict with the clusters the user
    already booked kept in the ranking."""
    rows = torch.as_tensor(graph.user_index.get_indexer(user_ids), device=DEVICE)
    return model.predict(rows, exclude_seen=False).float().cpu().numpy()


def rank(scores):
    """Clusters of each row ordered best first."""
    return np.argsort(-scores, axis=1, kind="stable")


scores = predict(users)
ranking = rank(scores)
positions = np.argsort(ranking, axis=1) + 1  # rank of every cluster, 1 = best
top = pd.DataFrame([[f"{c} ✓" if c in truth else c for c in ranked[:cfg.show_top]]
                    for ranked, truth in zip(ranking, labels)],
                   index=pd.Index(users, name="user_id"), columns=[f"#{r}" for r in range(1, cfg.show_top + 1)])
top.insert(0, "hạng của từng nhãn", [", ".join(f"{c} → #{pos[c]}" for c in sorted(truth, key=lambda c: pos[c]))
                                    for pos, truth in zip(positions, labels)])
print(f"Top {cfg.show_top} cluster model dự đoán (✓: cluster user book trong val)")
display(top)

Model: LightGCL, booking theo số lần + thời gian (GraphPro), λ2 = 1e-4 (data/runs/lightgcl_click_20261004_115153), epoch 7
Top 10 cluster model dự đoán (✓: cluster user book trong val)


,hạng của từng nhãn,#1,#2,#3,#4,#5,#6,#7,#8,#9,#10
user_id,,,,,,,,,,,
329118,38 → #23,95,41,68,72,70,98,21,25,18,97
373861,55 → #89,28,97,67,22,58,46,43,25,59,2
618548,"42 → #5, 49 → #27",91,48,9,98,42 ✓,18,95,41,28,72
775664,97 → #4,64,59,46,97 ✓,58,25,2,38,67,11
1029342,"42 → #1, 94 → #4",42 ✓,91,48,94 ✓,28,72,18,16,7,13


## 3. Kiểm chứng metric trên các user đã chọn


In [ ]:
METRICS = [f"{m}@{k}" for m in ("recall", "hit", "ndcg", "map") for k in cfg.topks]


def manual_metrics(ranked, truth, k):
    hits = [c in truth for c in ranked[:k]]
    dcg = sum(1 / math.log2(r + 1) for r, hit in enumerate(hits, start=1) if hit)
    idcg = sum(1 / math.log2(r + 1) for r in range(1, min(k, len(truth)) + 1))
    precision_sum = sum(sum(hits[:r]) / r for r, hit in enumerate(hits, start=1) if hit)
    return {f"recall@{k}": sum(hits) / len(truth), f"hit@{k}": float(any(hits)), f"ndcg@{k}": dcg / idcg,
            f"map@{k}": precision_sum / min(k, len(truth))}

def manual_table(user_ids, ranking, labels):
    rows = [{m: v for k in cfg.topks for m, v in manual_metrics(ranked.tolist(), truth, k).items()}
            for ranked, truth in zip(ranking, labels)]
    return pd.DataFrame(rows, index=pd.Index(user_ids, name="user_id"))[METRICS]


def compare(by_hand, by_code, code_label):
    gap = (by_hand - by_code).abs()
    if isinstance(gap, pd.DataFrame):
        by_hand, by_code, gap = by_hand.mean(), by_code.mean(), gap.max()
    display(fmt_table(pd.DataFrame({"tính tay": by_hand, code_label: by_code})))
    assert gap.max() < 1e-5, f"tính tay và {code_label} lệch nhau {gap.max():.1e}"


by_hand = manual_table(users, ranking, labels)
truth_dense = torch.as_tensor(val_split.labels[np.searchsorted(val_split.user_ids, users)].toarray() > 0)
code_metrics = topk_metrics(torch.as_tensor(scores), truth_dense, cfg.topks)
by_code = pd.DataFrame({m: code_metrics[m].numpy() for m in METRICS}, index=by_hand.index)
print("Metric tính tay, từng user")
display(fmt_table(by_hand))
compare(by_hand, by_code, "topk_metrics")

Metric tính tay, từng user


,recall@5,recall@10,recall@20,hit@5,hit@10,hit@20,ndcg@5,ndcg@10,ndcg@20,map@5,map@10,map@20
user_id,,,,,,,,,,,,
329118,0,0,0,0,0,0,0,0,0,0,0,0
373861,0,0,0,0,0,0,0,0,0,0,0,0
618548,"0,5","0,5","0,5",1,1,1,"0,2372","0,2372","0,2372","0,1","0,1","0,1"
775664,1,1,1,1,1,1,"0,4307","0,4307","0,4307","0,25","0,25","0,25"
1029342,1,1,1,1,1,1,"0,8772","0,8772","0,8772","0,75","0,75","0,75"


,tính tay,topk_metrics
recall@5,"0,5","0,5"
recall@10,"0,5","0,5"
recall@20,"0,5","0,5"
hit@5,"0,6","0,6"
hit@10,"0,6","0,6"
hit@20,"0,6","0,6"
ndcg@5,"0,309","0,309"
ndcg@10,"0,309","0,309"
ndcg@20,"0,309","0,309"
map@5,"0,22","0,22"


## 4. Kiểm chứng trên toàn bộ user val


In [13]:
EVAL_BATCH = max(256, 2**24 // n_items)


@torch.no_grad()
def evaluate(score_fn, split, users):
    """Mean metrics over the users of `split` that are in `users` (an Index of user ids), as in LightGCL_Final.ipynb;
    score_fn maps an array of user ids to a (users x items) score tensor."""
    keep = np.isin(split.user_ids, users)
    user_ids, labels = split.user_ids[keep], split.labels[keep]
    parts = []
    for start in range(0, len(user_ids), EVAL_BATCH):
        scores = torch.as_tensor(score_fn(user_ids[start:start + EVAL_BATCH])).float()
        batch_labels = torch.as_tensor(labels[start:start + EVAL_BATCH].toarray()) > 0
        parts.append(topk_metrics(scores, batch_labels, cfg.topks))
    return pd.Series({m: torch.cat([p[m] for p in parts]).mean().item() for m in METRICS})


all_ranking = np.concatenate([rank(predict(val_users[s:s + EVAL_BATCH]))[:, :max(cfg.topks)]
                              for s in range(0, len(val_users), EVAL_BATCH)])
all_by_hand = manual_table(val_users, all_ranking, val_labels(val_users))
compare(all_by_hand.mean(), evaluate(predict, val_split, graph.user_index), "evaluate()")
print(f"val recall@5 lưu lúc train: {trained['val recall@5']:.4f}, "
      f"tính lại: {all_by_hand['recall@5'].mean():.4f}")

,tính tay,evaluate()
recall@5,"0,2296","0,2296"
recall@10,"0,3518","0,3518"
recall@20,"0,5229","0,5229"
hit@5,"0,3187","0,3187"
hit@10,"0,4521","0,4521"
hit@20,"0,6171","0,6171"
ndcg@5,"0,1738","0,1738"
ndcg@10,"0,2171","0,2171"
ndcg@20,"0,2659","0,2659"
map@5,"0,1361","0,1361"


val recall@5 lưu lúc train: 0.2296, tính lại: 0.2296
